# 01 — Data Acquisition and Preparation

This notebook handles:
1. Loading the Chicago TNP (Transportation Network Providers) trip data
2. Loading US Census ACS demographic data by census tract
3. Merging trips with tract demographics
4. Cleaning and producing the analysis-ready dataset

---

## Data Sources

- **Chicago TNP Trips:** https://data.cityofchicago.org/Transportation/Transportation-Network-Providers-Trips-2018-2022-/m6dm-c72p  
  Export a filtered CSV from the portal (e.g., filter by a single quarter to keep size manageable — aim for ~500K–1M rows).  
  Place the downloaded file as `data/tnp_trips_raw.csv`.

- **US Census ACS 5-Year Estimates:** https://data.census.gov/  
  We need race and income by census tract for Cook County, IL (FIPS 17031).  
  This notebook includes code to download it via the Census API (no key needed for small requests),  
  or you can download manually and place as `data/census_tracts.csv`.

In [ ]:
import pandas as pd
import numpy as np
import requests
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)

print('Environment ready.')

## 1.1 — Load Chicago TNP Trip Data

The Chicago Data Portal allows you to export filtered CSVs directly.  
Alternatively, you can use the Socrata Open Data API (SODA) to pull a subset programmatically.

**Option A — Manual download (recommended):**  
Go to the portal link above → Filter by date (e.g., 2022-Q1) → Export CSV → save as `data/tnp_trips_raw.csv`.

**Option B — API download (below):**  
We query a manageable sample via the SODA API. The `$limit` and `$where` parameters control size.

In [ ]:
# ── Option B: Download via SODA API ──────────────────────────────────────────
# Pulls trips from a specific date range. Adjust dates and limit as needed.
# If you already placed the CSV manually, skip to the next cell.

SODA_ENDPOINT = 'https://data.cityofchicago.org/resource/m6dm-c72p.csv'

params = {
    '$where': "trip_start_timestamp >= '2022-01-01T00:00:00' AND trip_start_timestamp < '2022-04-01T00:00:00'",
    '$limit': 500000,
    '$select': ','.join([
        'trip_start_timestamp',
        'trip_seconds',
        'trip_miles',
        'fare',
        'tip',
        'additional_charges',
        'trip_total',
        'pickup_census_tract',
        'dropoff_census_tract',
        'pickup_community_area',
    ])
}

RAW_PATH = os.path.join(DATA_DIR, 'tnp_trips_raw.csv')

if not os.path.exists(RAW_PATH):
    print('Downloading trip data from Chicago Data Portal...')
    print('This may take a few minutes depending on connection speed.')
    resp = requests.get(SODA_ENDPOINT, params=params, timeout=300)
    resp.raise_for_status()
    with open(RAW_PATH, 'w') as f:
        f.write(resp.text)
    print(f'Saved to {RAW_PATH}')
else:
    print(f'Found existing file: {RAW_PATH}')

In [ ]:
# ── Load the raw trip data ───────────────────────────────────────────────────
trips_raw = pd.read_csv(RAW_PATH)
print(f'Loaded {len(trips_raw):,} rows, {trips_raw.shape[1]} columns')
trips_raw.head()

In [ ]:
trips_raw.info()
print('\n--- Missing values ---')
print(trips_raw.isnull().sum())

## 1.2 — Load Census Demographic Data

We need two tables from the ACS 5-Year Estimates for Cook County census tracts:
- **B02001** — Race (total, white alone, Black alone, etc.)
- **B19013** — Median Household Income

The Census API is free (no key needed for small requests, but a key gives higher rate limits).  
Get one at: https://api.census.gov/data/key_signup.html

In [ ]:
# ── Download Census ACS data ─────────────────────────────────────────────────
# Using 2021 ACS 5-Year (covers 2017-2021, aligns well with 2022 trip data)

CENSUS_PATH = os.path.join(DATA_DIR, 'census_tracts.csv')

if not os.path.exists(CENSUS_PATH):
    print('Downloading Census ACS data...')

    # Race data (B02001): total, white alone, Black alone, Asian alone
    race_url = (
        'https://api.census.gov/data/2021/acs/acs5'
        '?get=NAME,B02001_001E,B02001_002E,B02001_003E,B02001_005E'
        '&for=tract:*&in=state:17&in=county:031'
    )

    # Income data (B19013): median household income
    income_url = (
        'https://api.census.gov/data/2021/acs/acs5'
        '?get=NAME,B19013_001E'
        '&for=tract:*&in=state:17&in=county:031'
    )

    race_resp = requests.get(race_url, timeout=60)
    race_resp.raise_for_status()
    race_data = race_resp.json()
    race_df = pd.DataFrame(race_data[1:], columns=race_data[0])

    income_resp = requests.get(income_url, timeout=60)
    income_resp.raise_for_status()
    income_data = income_resp.json()
    income_df = pd.DataFrame(income_data[1:], columns=income_data[0])

    # Build the full 11-digit FIPS tract code: state(2) + county(3) + tract(6)
    for df in [race_df, income_df]:
        df['census_tract'] = df['state'] + df['county'] + df['tract']

    # Merge race and income
    census = race_df[['census_tract', 'B02001_001E', 'B02001_002E', 'B02001_003E', 'B02001_005E']].merge(
        income_df[['census_tract', 'B19013_001E']],
        on='census_tract'
    )

    census.columns = ['census_tract', 'total_pop', 'white_pop', 'black_pop', 'asian_pop', 'median_income']

    # Convert to numeric
    for col in ['total_pop', 'white_pop', 'black_pop', 'asian_pop', 'median_income']:
        census[col] = pd.to_numeric(census[col], errors='coerce')

    census.to_csv(CENSUS_PATH, index=False)
    print(f'Saved {len(census)} tracts to {CENSUS_PATH}')
else:
    census = pd.read_csv(CENSUS_PATH)
    print(f'Loaded {len(census)} census tracts')

census.head()

In [ ]:
# ── Derive sensitive group labels ────────────────────────────────────────────
# These are NOT model features — they are used only for auditing.

# Majority race label
census['pct_white'] = census['white_pop'] / census['total_pop']
census['pct_black'] = census['black_pop'] / census['total_pop']

def assign_majority_race(row):
    if pd.isna(row['total_pop']) or row['total_pop'] == 0:
        return 'unknown'
    if row['pct_white'] > 0.5:
        return 'majority_white'
    elif row['pct_black'] > 0.5:
        return 'majority_black'
    else:
        return 'mixed'

census['majority_race'] = census.apply(assign_majority_race, axis=1)

# Income group label (quartiles)
census['income_group'] = pd.qcut(
    census['median_income'].dropna(), q=4,
    labels=['low_income', 'lower_mid', 'upper_mid', 'high_income']
).reindex(census.index)

print('--- Majority race distribution ---')
print(census['majority_race'].value_counts())
print('\n--- Income group distribution ---')
print(census['income_group'].value_counts())

## 1.3 — Merge Trips with Census Data

In [ ]:
# ── Prepare trip data for merge ──────────────────────────────────────────────

trips = trips_raw.copy()

# Parse timestamp
trips['trip_start_timestamp'] = pd.to_datetime(trips['trip_start_timestamp'], errors='coerce')

# Ensure census tract is string for merge (the Chicago portal stores it as float)
trips['pickup_census_tract'] = (
    trips['pickup_census_tract']
    .dropna()
    .astype(float)
    .astype(int)
    .astype(str)
)

# Numeric columns
for col in ['trip_seconds', 'trip_miles', 'fare', 'tip', 'additional_charges', 'trip_total']:
    trips[col] = pd.to_numeric(trips[col], errors='coerce')

print(f'Trips before merge: {len(trips):,}')

In [ ]:
# ── Merge ────────────────────────────────────────────────────────────────────
census['census_tract'] = census['census_tract'].astype(str)

df = trips.merge(
    census[['census_tract', 'majority_race', 'income_group', 'median_income', 'pct_white', 'pct_black']],
    left_on='pickup_census_tract',
    right_on='census_tract',
    how='inner'
)

print(f'Trips after merge: {len(df):,}  (dropped {len(trips) - len(df):,} with no tract match)')

## 1.4 — Feature Engineering and Cleaning

In [ ]:
# ── Extract time features ────────────────────────────────────────────────────
df['hour'] = df['trip_start_timestamp'].dt.hour
df['day_of_week'] = df['trip_start_timestamp'].dt.dayofweek      # 0=Mon, 6=Sun
df['month'] = df['trip_start_timestamp'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)

In [ ]:
# ── Drop bad rows ────────────────────────────────────────────────────────────
len_before = len(df)

df = df.dropna(subset=['fare', 'trip_miles', 'trip_seconds'])
df = df[df['fare'] > 0]
df = df[df['trip_miles'] > 0.1]         # drop near-zero-distance trips
df = df[df['trip_seconds'] > 60]         # drop trips shorter than 1 minute
df = df[df['fare'] < 200]               # drop extreme outliers
df = df[df['majority_race'] != 'unknown']

print(f'Rows dropped during cleaning: {len_before - len(df):,}')
print(f'Final dataset: {len(df):,} rows')

In [ ]:
# ── Derived metric: price per mile ───────────────────────────────────────────
df['price_per_mile'] = df['fare'] / df['trip_miles']

print('--- Quick summary by majority race ---')
print(df.groupby('majority_race')[['fare', 'trip_miles', 'price_per_mile']].mean().round(2))

In [ ]:
# ── Define columns ───────────────────────────────────────────────────────────

FEATURE_COLS = ['trip_miles', 'trip_seconds', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour']
TARGET_COL = 'fare'
SENSITIVE_COL = 'majority_race'     # used for audit only, NOT as model input

# Verify no NaNs in features/target
assert df[FEATURE_COLS + [TARGET_COL, SENSITIVE_COL]].isnull().sum().sum() == 0, 'Still have NaNs!'
print('All clean. Columns for modeling:')
print(f'  Features:  {FEATURE_COLS}')
print(f'  Target:    {TARGET_COL}')
print(f'  Sensitive: {SENSITIVE_COL}')

In [ ]:
# ── Save the processed dataset ───────────────────────────────────────────────
PROCESSED_PATH = os.path.join(DATA_DIR, 'trips_processed.csv')

cols_to_save = FEATURE_COLS + [TARGET_COL, SENSITIVE_COL, 'income_group', 'median_income',
                                'pct_white', 'pct_black', 'price_per_mile', 'pickup_census_tract']
df[cols_to_save].to_csv(PROCESSED_PATH, index=False)

print(f'Saved processed dataset: {PROCESSED_PATH}  ({len(df):,} rows, {len(cols_to_save)} cols)')